In [0]:
# ============================================================
# Silver — Source 02: Debezium CDC
#
# Purpose: Capture every change to Postgres orders as an audit trail
# op=r (snapshot reads) are filtered out — already in Silver 01
# op=c (insert), op=u (update), op=d (delete) are kept
#
# Source:  bronze.src_02_cdc.events
# Target:  silver.src_02_cdc.order_changes
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable
import re as re_lib

BRONZE_CATALOG = 'bronze'
SILVER_CATALOG = 'silver'
SOURCE = 'src_02_cdc'
TARGET_TABLE = f'{SILVER_CATALOG}.src_02_cdc.order_changes'

spark.sql(f'CREATE SCHEMA IF NOT EXISTS {SILVER_CATALOG}.src_02_cdc')
print('Silver Source 02 CDC — starting...')


In [0]:
# ── LOAD BRONZE CDC ───────────────────────────────────────────
cdc_bronze = spark.table(f'{BRONZE_CATALOG}.src_02_cdc.events')
total = cdc_bronze.count()
print(f'Bronze CDC rows: {total}')

# Show op distribution
cdc_bronze.groupBy('cdc_op').count().show()

# Filter — only keep real changes (not snapshots)
# op=r means read/snapshot — redundant with Silver 01
# op=c = INSERT, op=u = UPDATE, op=d = DELETE
changes = cdc_bronze.filter(F.col('cdc_op').isin(['c', 'u', 'd']))
change_count = changes.count()
print(f'Live change events (op=c/u/d): {change_count}')

if change_count == 0:
    print('No live CDC changes yet — MSK not running.')
    print('This notebook will produce rows when MSK is recreated and generators run in stream mode.')
    print('Bronze 02 currently contains only op=r (snapshot) rows.')
    dbutils.notebook.exit('No live CDC changes — skipping Silver write')


In [0]:
# ── PARSE STRUCT STRING FORMAT ────────────────────────────────
# Debezium after_raw format: "item_id=1,order_id=2,product_sku=SKU-001,..."
# Extract key fields using regex

from pyspark.sql.functions import udf

@udf(returnType=LongType())
def extract_long(s, field):
    if not s: return None
    try:
        m = re_lib.search(rf'{field}=([-\d]+)', s)
        return int(m.group(1)) if m else None
    except: return None

@udf(returnType=StringType())
def extract_str(s, field):
    if not s: return None
    try:
        m = re_lib.search(rf'{field}=([^,}}]+)', s)
        return m.group(1).strip() if m else None
    except: return None

# Parse after_raw to get order fields
parsed = changes \
    .withColumn('order_id',    extract_long(F.col('after_raw'), F.lit('order_id'))) \
    .withColumn('customer_id', extract_long(F.col('after_raw'), F.lit('customer_id'))) \
    .withColumn('order_status', extract_str(F.col('after_raw'), F.lit('order_status'))) \
    .withColumn('total_pence',  extract_long(F.col('after_raw'), F.lit('total_amount_pence'))) \
    .withColumn('changed_at',   (F.col('cdc_ts_ms') / 1000).cast('timestamp')) \
    .select(
        'cdc_event_id',
        'cdc_op',
        'cdc_table',
        'changed_at',
        'order_id',
        'customer_id',
        'order_status',
        'total_pence',
        'after_raw',
    )

row_count = parsed.count()
print(f'Parsed CDC events: {row_count}')
parsed.groupBy('cdc_op', 'cdc_table').count().show()

# Write to Silver
if spark.catalog.tableExists(TARGET_TABLE):
    dt = DeltaTable.forName(spark, TARGET_TABLE)
    dt.alias('t').merge(parsed.alias('s'), 't.cdc_event_id = s.cdc_event_id') \
        .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    print(f'MERGE complete')
else:
    parsed.write.format('delta').mode('overwrite') \
        .option('mergeSchema', 'true').saveAsTable(TARGET_TABLE)
    print(f'Initial load complete')

print(f'\n✅ {TARGET_TABLE}: {row_count} rows')


In [0]:
# ── VERIFY ───────────────────────────────────────────────────
try:
    count = spark.sql(f'SELECT COUNT(*) as cnt FROM {TARGET_TABLE}').collect()[0]['cnt']
    print(f'{TARGET_TABLE}: {count} rows')
    spark.sql(f'SELECT cdc_op, cdc_table, changed_at, order_id, order_status FROM {TARGET_TABLE} LIMIT 5').show(truncate=False)
except:
    print('Table not created yet — no live CDC events in Bronze')
